# LATS (MCTS over actions)

> Notebook generated from [`examples/patterns/03_lats.py`](../../examples/patterns/03_lats.py).

| Field | Value |
|------|-------|
| **Dataset** | WebArena-style (embedded, 3 scenarios) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** MCTS tree with visit counts and UCB1 values; winning action path and final reward.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
LATS — Language Agent Tree Search (MCTS over agent actions)
=================================================================
Pattern: SPEC-PAT-004 / prismal.agents.patterns.lats

Dataset: WebArena / text planning tasks
  • Set of text-based navigation and planning tasks.
  • For this example we use synthetic planning tasks that simulate
    the action space of an information-seeking agent.
  • Inspired by: https://arxiv.org/abs/2307.13854 (WebArena)
  • Why: LATS/MCTS is ideal for discrete action spaces where you need
    to balance exploration and exploitation (UCB1). Multi-step planning
    tasks capture exactly this dynamic.

Pattern description:
  LATSAgent implements MCTS over agent actions:
  1. Select   — UCB1 to choose the most promising leaf node
  2. Expand   — generate candidate actions with action_generator
  3. Simulate — score the resulting state with reward_fn
  4. Backprop — propagate reward back up the tree

Required callables:
  - action_generator(state, tools) → list of candidate actions
  - transition_fn(state, action) → new state
  - reward_fn(state) → float [0, 1]

Usage:
    uv run python examples/patterns/03_lats.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
from typing import Any

from prismal.agents.patterns.lats import LATSAgent, LATSResult

## Dataset: text planning tasks

In [ ]:
# Synthetic tasks that simulate WebArena / ALFWorld style:
# the agent must navigate an action space to reach a goal.

PLANNING_TASKS = [
    {
        "id": "T1",
        "goal": "Find the cheapest flight price between Madrid and New York for January 15",
        "initial_state": {
            "location": "home",
            "info_gathered": [],
            "steps_taken": 0,
            "goal": "find_cheapest_flight_MAD_NYC",
        },
        "terminal_keyword": "price",
    },
    {
        "id": "T2",
        "goal": "Book a table for 4 at an Italian restaurant in Barcelona in fewer than 3 steps",
        "initial_state": {
            "location": "search_engine",
            "info_gathered": [],
            "steps_taken": 0,
            "goal": "book_restaurant_Barcelona",
        },
        "terminal_keyword": "reservation confirmed",
    },
    {
        "id": "T3",
        "goal": "Research the 3 best AI agent frameworks in 2024 and create a comparative summary",
        "initial_state": {
            "location": "search_engine",
            "info_gathered": [],
            "steps_taken": 0,
            "goal": "research_ai_agent_frameworks",
        },
        "terminal_keyword": "comparison",
    },
]

## Definition of available tools

In [ ]:
AVAILABLE_TOOLS = [
    {"name": "search_web", "description": "Search information on the internet"},
    {"name": "open_url", "description": "Open a specific URL"},
    {"name": "extract_info", "description": "Extract information from a page"},
    {"name": "compare_options", "description": "Compare multiple options"},
    {"name": "make_selection", "description": "Select the best option"},
    {"name": "confirm_action", "description": "Confirm and execute an action"},
    {"name": "summarize", "description": "Summarize gathered information"},
]

## Agent callables

In [ ]:
async def action_generator(state: dict[str, Any], tools: list[dict]) -> list[str]:
    """Generate candidate actions given the current state.

    In production this would call the LLM. Here we use state-based
    heuristics to simulate the WebArena action space.

    Args:
        state: Current agent state.
        tools: Available tools.

    Returns:
        List of candidate actions as strings.
    """
    steps = state.get("steps_taken", 0)
    goal = state.get("goal", "")
    info = state.get("info_gathered", [])

    # Simulate generation of contextual actions
    if steps == 0:
        return [
            f"search_web(query='{goal}')",
            f"search_web(query='{goal} price best option')",
            f"search_web(query='{goal} 2024 comparison')",
        ]
    if steps == 1 and not info:
        return [
            "extract_info(source='first result')",
            "extract_info(source='second result')",
            "open_url(url='most relevant result')",
        ]
    if steps == 2:
        return [
            "compare_options(results='all results')",
            "summarize(content='gathered information')",
            "make_selection(criterion='best value')",
        ]
    return [
        "confirm_action(selection='chosen option')",
        "summarize(content='final result with price/reservation/comparison')",
    ]


async def transition_fn(state: dict[str, Any], action: str) -> dict[str, Any]:
    """Apply an action to the state and return the new state.

    Simulates the state transition of a planning agent.

    Args:
        state: Current state.
        action: Action to execute.

    Returns:
        New state after executing the action.
    """
    new_state = dict(state)
    new_info = list(state.get("info_gathered", []))

    # Simulate the effect of each tool
    if "search_web" in action:
        new_info.append(f"search_results:{action[:50]}")
        new_state["location"] = "search_results"
    elif "extract_info" in action:
        new_info.append("extracted_info:price_comparison_options")
        new_state["location"] = "info_page"
    elif "compare_options" in action:
        new_info.append("comparison:option_A_best_price_option_B_best_service")
        new_state["location"] = "comparison_view"
    elif "summarize" in action:
        goal = state.get("goal", "")
        if "flight" in goal:
            new_info.append("summary:cheapest_flight_price_450EUR_airline_X")
        elif "restaurant" in goal:
            new_info.append("summary:reservation_confirmed_Italian_restaurant_20:00")
        elif "frameworks" in goal:
            new_info.append("summary:comparison_LangGraph_vs_AutoGen_vs_CrewAI")
    elif "confirm_action" in action:
        new_info.append("action_confirmed:task_completed")
        new_state["location"] = "confirmation"

    new_state["info_gathered"] = new_info
    new_state["steps_taken"] = state.get("steps_taken", 0) + 1
    new_state["last_action"] = action
    return new_state


async def reward_fn(state: dict[str, Any]) -> float:
    """Score the current state in [0, 1].

    Reward criteria:
    - Useful accumulated information (+0.2 per relevant item)
    - Efficient steps (fewer steps → better reward)
    - Task completed (+0.5 bonus)

    Args:
        state: State to evaluate.

    Returns:
        Reward in [0.0, 1.0].
    """
    info = state.get("info_gathered", [])
    steps = state.get("steps_taken", 0)
    state.get("goal", "")

    score = 0.0

    # Reward for relevant accumulated information
    relevant_keywords = {"price", "comparison", "confirmed", "summary", "options"}
    for item in info:
        if any(kw in item.lower() for kw in relevant_keywords):
            score += 0.2

    # Penalty for too many steps (efficiency)
    if steps > 5:
        score -= 0.1 * (steps - 5)

    # Bonus for completing the task
    terminal_markers = {"confirmed", "price_", "comparison_"}
    for item in info:
        if any(m in item for m in terminal_markers) and "summary" in item:
            score += 0.5
            break

    return max(0.0, min(1.0, score))


## Main function

In [ ]:
async def run_lats_task(task: dict) -> LATSResult:
    """Run LATS on a planning task.

    Args:
        task: Dictionary with goal, initial_state and metadata.

    Returns:
        LATSResult with the best action sequence found.
    """
    agent = LATSAgent(
        tools=AVAILABLE_TOOLS,
        reward_fn=reward_fn,
        action_generator=action_generator,
        transition_fn=transition_fn,
        max_simulations=30,  # MCTS simulations
        exploration_constant=1.41,  # √2 (Auer et al. 2002)
        max_depth=5,  # maximum tree depth
        timeout_seconds=30.0,  # safety timeout
        terminal_reward=0.95,  # early-stop threshold
    )

    return await agent.search(
        initial_state=task["initial_state"],
        goal=task["goal"],
    )


async def main() -> None:
    print("=" * 70)
    print("  LATS (MCTS) — Dataset: WebArena-style planning tasks")
    print("=" * 70)

    for task in PLANNING_TASKS:
        print(f"\n[Task {task['id']}]")
        print(f"  Goal: {task['goal']}")

        result = await run_lats_task(task)

        print(f"\n  Simulations: {result.total_simulations}")
        print(f"  Nodes explored: {result.nodes_explored}")
        print(f"  Best reward: {result.best_reward:.3f}")
        print(f"  Task completed: {'Yes' if result.goal_reached else 'No'}")

        print("\n  Best action sequence:")
        for j, action in enumerate(result.best_action_sequence, 1):
            print(f"    {j}. {action}")

        if result.best_state:
            info = result.best_state.get("info_gathered", [])
            print(f"\n  Gathered information ({len(info)} items):")
            for item in info:
                print(f"    • {item}")

        print("-" * 70)

    print("\n[UCB1 in action]")
    print("  The exploration coefficient C=√2 balances:")
    print("  • Exploitation: nodes with high mean reward (Q/N)")
    print("  • Exploration: rarely visited nodes (√(ln N_parent / N))")
    print("  Result: efficient coverage of the action space")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()